# 🦾 Robotic Arm — Curriculum PPO (Crash-Safe Version)

**What's changed from the original:**
- `matplotlib.use('Agg')` — no GUI backend, plots never crash the kernel
- `CSVLoggerCallback` — every episode flushes reward to `training_log.csv` (survives crashes)
- `training_manifest.json` — written to disk at the end of each stage
- Plotting loads from CSV (not memory); uses last 1000 pts + `[::10]` downsampling; smoothed curve only; small figure; `plt.savefig()`
- Stage-wise subplots kept intact
- No `plt.show()` anywhere

In [2]:
# Cell 1 — Install panda-gym
!pip install panda-gym -q

In [3]:
# Cell 2 — Quick sanity check
import panda_gym
import gymnasium as gym

env = gym.make("PandaReachDense-v3")
env.close()
print("✅ Working")

✅ Working


In [4]:
# Cell 3 — Install any missing packages
import importlib
import subprocess
import sys

def install_if_missing(pkg_name, import_name=None, extra_args=""):
    try:
        importlib.import_module(import_name or pkg_name)
        print(f"✅ {pkg_name} already installed")
    except ImportError:
        print(f"⬇️  Installing {pkg_name}...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q"] + pkg_name.split() + extra_args.split(),
            check=True
        )

install_if_missing("torch", "torch")
install_if_missing("stable-baselines3[extra]", "stable_baselines3")
install_if_missing("panda-gym pybullet", "panda_gym")
install_if_missing("gymnasium", "gymnasium")
install_if_missing("tensorboard", "tensorboard")
install_if_missing("matplotlib", "matplotlib")
install_if_missing("seaborn", "seaborn")
install_if_missing("tqdm", "tqdm")
install_if_missing("pandas", "pandas")

print("\n✅ Setup complete")

✅ torch already installed
✅ stable-baselines3[extra] already installed
✅ panda-gym pybullet already installed
✅ gymnasium already installed
✅ tensorboard already installed
✅ matplotlib already installed
✅ seaborn already installed
✅ tqdm already installed
✅ pandas already installed

✅ Setup complete


In [5]:
# Cell 4 — Imports and global config
# ── SAFETY: Use non-interactive Agg backend BEFORE importing pyplot ──────────
import matplotlib
matplotlib.use('Agg')   # No GUI window — prevents ALL kernel crashes from plt.show()

import os, time, csv, warnings, json
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import gymnasium as gym
from gymnasium import spaces
import panda_gym  # noqa — registers envs

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize, SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import (
    CheckpointCallback, EvalCallback, CallbackList, BaseCallback
)
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed

warnings.filterwarnings("ignore")

# ── Dirs ─────────────────────────────────────────────────────────────────────
GRAPH_DIR = "./ppt_graphs"
SAVE_DIR  = "./curriculum_models"
LOG_DIR   = "./logs"
for d in [GRAPH_DIR, SAVE_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Plot style — lightweight, crash-safe ─────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":        80,       # REDUCED from 150 — saves memory
    "savefig.dpi":       150,      # REDUCED from 300 — saves memory
    "figure.facecolor":  "white",
    "axes.facecolor":    "#f5f6fa",
    "axes.grid":         True,
    "grid.alpha":        0.35,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
    "axes.titlesize":    12,
    "axes.titleweight":  "bold",
    "axes.labelsize":    10,
    "legend.fontsize":   9,
})

# Stage colour palette — used consistently in every graph
STAGE_COLORS = {
    "Stage 1 — Reach":       "#1a3a6b",
    "Stage 2 — Grasp/Lift":  "#e05c00",
    "Stage 3 — Place":       "#2d8a4e",
}
STAGE_LIST = list(STAGE_COLORS.keys())

def save_fig(fig, fname):
    """Save figure to GRAPH_DIR, then close it. Never calls plt.show()."""
    path = os.path.join(GRAPH_DIR, fname)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)   # ← frees memory immediately
    print(f"  💾 Saved → {path}")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cpu"   # Force CPU for stability

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"⚠️  GPU detected ({gpu_name}, {gpu_mem:.1f} GB) but NOT used (stability)")
else:
    print("ℹ️  No CUDA — using CPU")

print("✅ Imports and config ready. Backend:", matplotlib.get_backend())

⚠️  GPU detected (NVIDIA GeForce RTX 4060 Laptop GPU, 8.6 GB) but NOT used (stability)
✅ Imports and config ready. Backend: Agg


In [5]:
# Cell 5 — Reward Wrappers (all 3 stages)

class ReachRewardWrapper(gym.Wrapper):
    """
    Stage 1 — REACH
    Reward = -dist(end_effector, object)
    """
    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        ee_pos  = obs["observation"][:3]
        obj_pos = obs["observation"][6:9]
        dist    = float(np.linalg.norm(ee_pos - obj_pos))
        reward  = -dist
        info["is_success"] = float(dist < 0.05)
        return obs, reward, terminated, truncated, info


class GraspLiftRewardWrapper(gym.Wrapper):
    """
    Stage 2 — GRASP / LIFT
    Reward = reach component + large bonus for lifting object above table.
    """
    LIFT_THRESHOLD = 0.05
    LIFT_BONUS     = 2.0

    def __init__(self, env):
        super().__init__(env)
        self._table_z = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._table_z = obs["observation"][8]
        return obs, info

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        ee_pos       = obs["observation"][:3]
        obj_pos      = obs["observation"][6:9]
        reach_reward = -float(np.linalg.norm(ee_pos - obj_pos))
        obj_z        = obj_pos[2]
        table_z      = self._table_z if self._table_z is not None else obj_z
        lifted       = obj_z > (table_z + self.LIFT_THRESHOLD)
        lift_reward  = self.LIFT_BONUS if lifted else 0.0
        reward       = reach_reward + lift_reward
        info["is_success"] = float(lifted)
        info["lifted"]     = lifted
        return obs, reward, terminated, truncated, info


class PlaceRewardWrapper(gym.Wrapper):
    """
    Stage 3 — PLACE (full task)
    Uses env's native dense reward + small lift shaping bonus.
    """
    LIFT_THRESHOLD = 0.04

    def __init__(self, env):
        super().__init__(env)
        self._table_z = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._table_z = obs["observation"][8]
        return obs, info

    def step(self, action):
        obs, env_reward, terminated, truncated, info = self.env.step(action)
        obj_z   = obs["observation"][8]
        table_z = self._table_z if self._table_z is not None else obj_z
        lifted  = obj_z > (table_z + self.LIFT_THRESHOLD)
        shaping = 0.3 if lifted else 0.0
        reward  = env_reward + shaping
        return obs, reward, terminated, truncated, info


print("✅ Reward wrappers defined:")
print("   ReachRewardWrapper      → Stage 1")
print("   GraspLiftRewardWrapper  → Stage 2")
print("   PlaceRewardWrapper      → Stage 3")

✅ Reward wrappers defined:
   ReachRewardWrapper      → Stage 1
   GraspLiftRewardWrapper  → Stage 2
   PlaceRewardWrapper      → Stage 3


In [ ]:
# Cell 6 — Curriculum definition
BASE_ENV_ID = "PandaPickAndPlaceDense-v3"
N_ENVS      = 1

CURRICULUM = [
    {
        "name":       "Stage 1 — Reach",
        "wrapper":    ReachRewardWrapper,
        "timesteps":  50_000,
        "ent_coef":   0.02,
        "lr":         3e-4,
        "save_name":  "stage1_reach",
        "log_subdir": os.path.join(LOG_DIR, "stage1"),
    },
    {
        "name":       "Stage 2 — Grasp/Lift",
        "wrapper":    GraspLiftRewardWrapper,
        "timesteps":  75_000,
        "ent_coef":   0.01,
        "lr":         1e-4,
        "save_name":  "stage2_grasp",
        "log_subdir": os.path.join(LOG_DIR, "stage2"),
    },
    {
        "name":       "Stage 3 — Place",
        "wrapper":    PlaceRewardWrapper,
        "timesteps":  100_000,
        "ent_coef":   0.005,
        "lr":         5e-5,
        "save_name":  "stage3_place_FINAL",
        "log_subdir": os.path.join(LOG_DIR, "stage3"),
    },
]

for s in CURRICULUM:
    os.makedirs(s["log_subdir"], exist_ok=True)

total = sum(s["timesteps"] for s in CURRICULUM)
print(f"📋 Curriculum — {len(CURRICULUM)} stages | {total:,} total steps")
for s in CURRICULUM:
    print(f"   {s['name']:28s}  {s['timesteps']:>7,} steps  "
          f"| lr={s['lr']}  ent={s['ent_coef']}")

📋 Curriculum — 3 stages | 225,000 total steps
   Stage 1 — Reach                50,000 steps  | lr=0.0003  ent=0.02
   Stage 2 — Grasp/Lift           75,000 steps  | lr=0.0001  ent=0.01
   Stage 3 — Place               100,000 steps  | lr=5e-05  ent=0.005


In [7]:
# Cell 7 — Env factory + callbacks + CSVLoggerCallback

def load_monitor(log_dir):
    """Read Monitor CSV files from a log directory."""
    rewards, lengths = [], []
    for fname in os.listdir(log_dir):
        if fname.endswith(".monitor.csv"):
            fpath = os.path.join(log_dir, fname)
            with open(fpath, "r") as f:
                lines = f.readlines()
            data_lines = [l for l in lines if not l.startswith("#")]
            reader = csv.DictReader(data_lines)
            for row in reader:
                try:
                    rewards.append(float(row["r"]))
                    lengths.append(int(row["l"]))
                except (KeyError, ValueError):
                    pass
    return np.array(rewards), np.array(lengths)


def build_train_env(stage: dict, n_envs: int, seed: int):
    log_dir      = stage["log_subdir"]
    WrapperClass = stage["wrapper"]

    def make_env_fn(rank):
        def _init():
            env = gym.make(BASE_ENV_ID)
            env = WrapperClass(env)
            env = Monitor(env, filename=os.path.join(log_dir, f"env_{rank}"))
            env.reset(seed=seed + rank)
            return env
        set_random_seed(seed)
        return _init

    vec = DummyVecEnv([make_env_fn(i) for i in range(n_envs)])
    vec = VecNormalize(vec, norm_obs=True, norm_reward=True,
                       clip_obs=10.0, gamma=0.95)
    return vec


def build_eval_env(stage: dict, seed: int):
    WrapperClass = stage["wrapper"]

    def _init():
        env = gym.make(BASE_ENV_ID)
        env = WrapperClass(env)
        return env

    vec = DummyVecEnv([_init])
    vec = VecNormalize(vec, norm_obs=True, norm_reward=False, training=False)
    return vec


class StageProgressCallback(BaseCallback):
    """Compact progress bar printed every `log_every` steps."""

    def __init__(self, stage_name: str, total: int, log_every: int = 25_000):
        super().__init__(verbose=0)
        self.stage_name = stage_name
        self.total      = total
        self.log_every  = log_every
        self._last      = 0
        self._t0        = time.time()

    def _on_step(self) -> bool:
        if self.num_timesteps - self._last >= self.log_every:
            e   = time.time() - self._t0
            sps = self.num_timesteps / max(e, 1)
            eta = (self.total - self.num_timesteps) / max(sps, 1)
            pct = 100 * self.num_timesteps / self.total
            bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
            vm  = f" | VRAM {torch.cuda.memory_allocated()/1e6:.0f}MB" \
                  if DEVICE == "cuda" else ""
            print(f"  [{bar}] {pct:5.1f}%  "
                  f"{self.num_timesteps:>7,}/{self.total:,}  "
                  f"| {sps:,.0f} sps | ETA {eta/60:.1f} min{vm}")
            self._last = self.num_timesteps
        return True


# ── CSV Logger Callback ───────────────────────────────────────────────────────
class CSVLoggerCallback(BaseCallback):
    """
    Logs one row per completed episode to training_log.csv.
    Format: [stage_name, episode_number, reward]
    Also logs ep_length and timestep for downstream plotting.
    Flushes immediately — data is safe even if kernel crashes mid-training.
    Extracts reward from info["episode"]["r"] as set by SB3's Monitor wrapper.
    """

    # Shared log file path written by ALL stages (append mode per stage)
    SHARED_CSV = "./training_log.csv"

    def __init__(self, csv_path: str, stage_name: str, verbose=0):
        super().__init__(verbose)
        self.csv_path   = csv_path       # per-stage CSV (detailed)
        self.stage_name = stage_name
        self._ep_count  = 0
        self._file      = None
        self._writer    = None
        self._shared_file   = None
        self._shared_writer = None

    def _on_training_start(self):
        # Per-stage CSV (detailed columns)
        self._file   = open(self.csv_path, "w", newline="")
        self._writer = csv.writer(self._file)
        self._writer.writerow(["episode", "timestep", "reward", "ep_length", "stage"])
        self._file.flush()

        # Shared training_log.csv — append if exists (required format: stage_name, episode, reward)
        write_header = not os.path.exists(self.SHARED_CSV)
        self._shared_file   = open(self.SHARED_CSV, "a", newline="")
        self._shared_writer = csv.writer(self._shared_file)
        if write_header:
            self._shared_writer.writerow(["stage_name", "episode_number", "reward"])
            self._shared_file.flush()

    def _on_step(self) -> bool:
        # SB3 Monitor sets info["episode"] = {"r": total_reward, "l": length, "t": time}
        for info in self.locals.get("infos", []):
            if "episode" in info:
                ep_info = info["episode"]
                self._ep_count += 1
                reward    = round(float(ep_info["r"]), 4)
                ep_length = int(ep_info["l"])

                # Per-stage CSV
                self._writer.writerow([
                    self._ep_count,
                    self.num_timesteps,
                    reward,
                    ep_length,
                    self.stage_name,
                ])
                self._file.flush()  # ← survives kernel crash

                # Shared training_log.csv [stage_name, episode_number, reward]
                self._shared_writer.writerow([
                    self.stage_name,
                    self._ep_count,
                    reward,
                ])
                self._shared_file.flush()  # ← survives kernel crash
        return True

    def _on_training_end(self):
        if self._file:
            self._file.close()
        if self._shared_file:
            self._shared_file.close()
        print(f"  💾 CSV log saved → {self.csv_path}")
        print(f"  💾 Shared log   → {self.SHARED_CSV}")


# ── Manifest helpers ──────────────────────────────────────────────────────────
MANIFEST_PATH = os.path.join(LOG_DIR, "training_manifest.json")

def save_training_manifest(stage_name, save_name, csv_path, eval_log_path, color):
    """Append/update a stage entry in the training manifest JSON."""
    manifest = {}
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            manifest = json.load(f)
    manifest[stage_name] = {
        "save_name":  save_name,
        "csv_path":   csv_path,
        "eval_log":   eval_log_path,
        "color":      color,
    }
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"  📋 Manifest updated → {MANIFEST_PATH}")


print("✅ Env factory and callbacks ready.")

✅ Env factory and callbacks ready.


In [8]:
# Cell 8 — Curriculum Training Loop
# PPO.learn() is UNCHANGED. CSVLoggerCallback is injected via CallbackList.
SEED = 42

POLICY_KWARGS = dict(
    net_arch      = dict(pi=[256, 256], vf=[256, 256]),
    activation_fn = torch.nn.Tanh,
)

# stage_history kept in memory as fallback; manifest on disk is the safe copy
stage_history   = {}
prev_model_path = None

print("═" * 65)
print("  🎓  CURRICULUM TRAINING")
print(f"  Base env : {BASE_ENV_ID}")
print(f"  Device   : {DEVICE.upper()}  |  {N_ENVS} parallel envs")
print("═" * 65)

wall_t0 = time.time()

for stage_idx, stage in enumerate(CURRICULUM):
    sname  = stage["name"]
    tsteps = stage["timesteps"]
    color  = list(STAGE_COLORS.values())[stage_idx]

    print(f"\n{'─'*65}")
    print(f"  ▶  {sname}  ({tsteps:,} steps)")
    print(f"{'─'*65}")

    train_env = build_train_env(stage, N_ENVS, SEED)
    eval_env  = build_eval_env(stage, SEED)

    if prev_model_path is None:
        model = PPO(
            policy          = "MultiInputPolicy",
            env             = train_env,
            device          = DEVICE,
            learning_rate   = stage["lr"],
            n_steps         = 1024,
            batch_size      = 256,
            n_epochs        = 10,
            gamma           = 0.95,
            gae_lambda      = 0.95,
            clip_range      = 0.2,
            ent_coef        = stage["ent_coef"],
            vf_coef         = 0.5,
            max_grad_norm   = 0.5,
            policy_kwargs   = POLICY_KWARGS,
            verbose         = 0,
            tensorboard_log = stage["log_subdir"],
            seed            = SEED,
        )
        n_params = sum(p.numel() for p in model.policy.parameters())
        print(f"  New model | {n_params:,} parameters")
    else:
        model = PPO.load(
            prev_model_path,
            env    = train_env,
            device = DEVICE,
            custom_objects = {
                "learning_rate": stage["lr"],
                "ent_coef":      stage["ent_coef"],
                "clip_range":    0.2,
            }
        )
        print(f"  ✅ Weights transferred from {prev_model_path}")

    stage_save_dir = os.path.join(SAVE_DIR, f"stage{stage_idx+1}")
    os.makedirs(stage_save_dir, exist_ok=True)

    ckpt_cb = CheckpointCallback(
        save_freq   = max(tsteps // 4, 10_000),
        save_path   = stage_save_dir,
        name_prefix = f"ppo_s{stage_idx+1}",
        verbose     = 0,
    )

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path = stage_save_dir,
        log_path             = stage["log_subdir"],
        eval_freq            = 5000,
        n_eval_episodes      = 10,
        deterministic        = True,
        render               = False,
        verbose              = 0,
    )

    prog_cb = StageProgressCallback(sname, tsteps, log_every=25_000)

    # CSV logger — writes to disk every episode, flushes immediately
    # Logging happens DURING training (inside model.learn via callback)
    csv_path = os.path.join(LOG_DIR, f"training_log_{stage['save_name']}.csv")
    csv_cb   = CSVLoggerCallback(csv_path, stage_name=sname)

    t0 = time.time()
    try:
        # ── PPO.learn() is UNCHANGED ──────────────────────────────────────
        model.learn(
            total_timesteps     = tsteps,
            callback            = CallbackList([ckpt_cb, eval_cb, prog_cb, csv_cb]),
            reset_num_timesteps = True,
            log_interval        = 1,
        )
        elapsed = time.time() - t0
        print(f"  ✅ {sname} done in {elapsed/60:.1f} min")

    except KeyboardInterrupt:
        print(f"  ⏸  Interrupted during {sname} — saving checkpoint ...")

    finally:
        final_path = os.path.join(SAVE_DIR, stage["save_name"])
        model.save(final_path)
        train_env.save(final_path + "_vecnorm.pkl")
        prev_model_path = final_path + ".zip"

        print(f"  💾 Model   → {final_path}.zip")
        print(f"  💾 VecNorm → {final_path}_vecnorm.pkl")

        eval_path = os.path.join(stage["log_subdir"], "evaluations.npz")

        stage_history[sname] = {
            "log_dir":  stage["log_subdir"],
            "color":    color,
            "eval_log": eval_path,
            "csv_path": csv_path,
        }

        # Write manifest to disk immediately after each stage
        save_training_manifest(
            stage_name     = sname,
            save_name      = stage["save_name"],
            csv_path       = csv_path,
            eval_log_path  = eval_path,
            color          = color,
        )

        if os.path.exists(eval_path):
            data = np.load(eval_path, allow_pickle=True)
            timesteps_arr = data["timesteps"]
            results       = data["results"]
            mean_rewards  = results.mean(axis=1)
            final_reward  = mean_rewards[-1]

            success_rate = None
            if "successes" in data:
                success_rate = data["successes"].mean()

            try:
                _r, _lengths = load_monitor(stage["log_subdir"])
                ep_len = _lengths[-50:].mean() if len(_lengths) > 0 else None
            except Exception:
                ep_len = None

            print(f"\n📊 FINAL METRICS — {sname}")
            print(f"   Timesteps      : {timesteps_arr[-1]}")
            print(f"   Mean Reward    : {final_reward:.3f}")
            if success_rate is not None:
                print(f"   Success Rate   : {success_rate:.2f}")
            if ep_len is not None:
                print(f"   Avg Ep Length  : {ep_len:.1f} steps")
        else:
            print(f"   ℹ️  No eval log yet (eval_freq > timesteps?)")

        train_env.close()
        eval_env.close()

total_wall = time.time() - wall_t0
print(f"\n{'═'*65}")
print(f"  🏁  All stages complete in {total_wall/60:.1f} min")
print(f"{'═'*65}")

═════════════════════════════════════════════════════════════════
  🎓  CURRICULUM TRAINING
  Base env : PandaPickAndPlaceDense-v3
  Device   : CPU  |  1 parallel envs
═════════════════════════════════════════════════════════════════

─────────────────────────────────────────────────────────────────
  ▶  Stage 1 — Reach  (50,000 steps)
─────────────────────────────────────────────────────────────────
  New model | 146,185 parameters
  [██████████░░░░░░░░░░]  50.0%   25,000/50,000  | 426 sps | ETA 1.0 min
  [████████████████████] 100.0%   50,000/50,000  | 441 sps | ETA 0.0 min
  💾 CSV log saved → ./logs\training_log_stage1_reach.csv
  💾 Shared log   → ./training_log.csv
  ✅ Stage 1 — Reach done in 1.9 min
  💾 Model   → ./curriculum_models\stage1_reach.zip
  💾 VecNorm → ./curriculum_models\stage1_reach_vecnorm.pkl
  📋 Manifest updated → ./logs\training_manifest.json

📊 FINAL METRICS — Stage 1 — Reach
   Timesteps      : 50000
   Mean Reward    : -4.478
   Success Rate   : 0.33
   Avg Ep L

## 📊 Plotting — runs independently, loads from disk

If your kernel crashed after training: **restart the kernel, run cells 1–4 only** (imports), then run from Cell 9 onward. No retraining needed.

All plots:
- Load data from `training_log.csv` / per-stage CSVs (not in-memory arrays)
- Downsample to last 1000 points with `[::10]` — minimal memory
- Show **smoothed curve only** (no raw scatter)
- Use small figure sizes
- Save with `plt.savefig()` — never `plt.show()`

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

LOG_DIR       = "./logs"
GRAPH_DIR     = "./ppt_graphs"
SAVE_DIR      = "./curriculum_models"
MANIFEST_PATH = os.path.join(LOG_DIR, "training_manifest.json")

print("✅ All variables defined")
print(f"   MANIFEST_PATH = {MANIFEST_PATH}")
print(f"   Exists on disk: {os.path.exists(MANIFEST_PATH)}")

✅ All variables defined
   MANIFEST_PATH = ./logs\training_manifest.json
   Exists on disk: True


In [7]:
# Cell 9 — Load training data from disk (crash-safe, no memory dependency)
#
# Loads from:
#   1. training_manifest.json  — index of all stages written during training
#   2. training_log_<stage>.csv — per-episode reward + length from CSVLoggerCallback
#   3. evaluations.npz         — EvalCallback eval snapshots (if they exist)
#
# Falls back to in-memory stage_history if manifest doesn't exist yet.

stage_data = {}

# ── Try loading from manifest (disk) first ────────────────────────────────────
if os.path.exists(MANIFEST_PATH):
    print(f"📂 Loading from manifest: {MANIFEST_PATH}")
    with open(MANIFEST_PATH, "r") as f:
        manifest = json.load(f)

    for sname, info in manifest.items():
        csv_path = info["csv_path"]
        if not os.path.exists(csv_path):
            print(f"  ⚠️  CSV missing for {sname}: {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        if df.empty:
            print(f"  ⚠️  Empty CSV for {sname}")
            continue

        stage_data[sname] = {
            "rewards":  df["reward"].values,
            "lengths":  df["ep_length"].values,
            "color":    info["color"],
            "eval_log": info["eval_log"],
        }
        print(f"  ✅ {sname}: {len(df)} episodes | "
              f"reward range [{df['reward'].min():.2f}, {df['reward'].max():.2f}]")

# ── Fallback: in-memory stage_history (if training just finished cleanly) ─────
elif 'stage_history' in dir() and stage_history:
    print("📂 No manifest found — using in-memory stage_history")
    for sname, info in stage_history.items():
        csv_path  = info.get("csv_path", "")
        eval_path = info["eval_log"]

        rewards, lengths = np.array([]), np.array([])

        # Try CSV first
        if csv_path and os.path.exists(csv_path):
            df      = pd.read_csv(csv_path)
            rewards = df["reward"].values
            lengths = df["ep_length"].values
        # Fall back to evaluations.npz
        elif os.path.exists(eval_path):
            data    = np.load(eval_path)
            rewards = data["results"].flatten()[-3000:]
            lengths = data["ep_lengths"].flatten()[-5000:]

        if len(rewards) == 0:
            print(f"  ⚠️  No data for {sname} — skipping")
            continue

        stage_data[sname] = {
            "rewards":  rewards,
            "lengths":  lengths,
            "color":    info["color"],
            "eval_log": eval_path,
        }
        print(f"  ✅ {sname}: {len(rewards)} episodes")

else:
    print("⚠️  No manifest and no in-memory data found.")
    print("   → Run training (Cell 8) first, or check that LOG_DIR is correct.")

if stage_data:
    print(f"\n📦 Ready to plot: {list(stage_data.keys())}")
else:
    print("\n❌ stage_data is empty — plotting cells will be skipped safely.")

📂 Loading from manifest: ./logs\training_manifest.json
  ✅ Stage 1 — Reach: 1056 episodes | reward range [-37.99, -0.07]
  ✅ Stage 2 — Grasp/Lift: 1574 episodes | reward range [-42.36, 84.17]
  ✅ Stage 3 — Place: 2098 episodes | reward range [-62.56, 8.61]

📦 Ready to plot: ['Stage 1 — Reach', 'Stage 2 — Grasp/Lift', 'Stage 3 — Place']


In [8]:
import psutil, os
proc = psutil.Process(os.getpid())
ram = psutil.virtual_memory()
print(f"Total RAM:     {ram.total / 1e9:.1f} GB")
print(f"Available RAM: {ram.available / 1e9:.1f} GB")
print(f"Used RAM:      {ram.used / 1e9:.1f} GB  ({ram.percent}%)")
print(f"This process:  {proc.memory_info().rss / 1e9:.2f} GB")

Total RAM:     16.9 GB
Available RAM: 5.0 GB
Used RAM:      11.8 GB  (70.1%)
This process:  0.57 GB


In [9]:
import gc, os
import numpy as np
from PIL import Image, ImageDraw

def smooth(y, alpha=0.05):
    if len(y) == 0:
        return np.array([])
    s = [float(y[0])]
    for v in y[1:]:
        s.append(alpha * float(v) + (1 - alpha) * s[-1])
    return np.array(s)

def save_line_plot_pil(values, title, out_path, color=(70, 130, 180)):
    W, H, PAD = 400, 250, 30
    img  = Image.new("RGB", (W, H), "white")
    draw = ImageDraw.Draw(img)

    mn, mx = values.min(), values.max()
    rng = mx - mn if mx != mn else 1.0

    # Normalize to canvas
    pts = []
    for i, v in enumerate(values):
        x = PAD + int(i * (W - 2*PAD) / max(len(values)-1, 1))
        y = H - PAD - int((v - mn) / rng * (H - 2*PAD))
        pts.append((x, y))

    # Draw axes
    draw.line([(PAD, PAD), (PAD, H-PAD), (W-PAD, H-PAD)], fill="black", width=1)
    # Draw line
    for i in range(len(pts)-1):
        draw.line([pts[i], pts[i+1]], fill=color, width=2)
    # Title
    draw.text((W//2 - len(title)*3, 8), title, fill="black")

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    img.save(out_path)
    del img, draw, pts, values
    gc.collect()
    print(f"✅ Saved: {out_path}")

if stage_data:
    colors_rgb = [(70,130,180), (255,140,0), (60,179,113)]
    for i, (sname, d) in enumerate(stage_data.items()):
        r = np.array(d["rewards"], dtype=np.float32)
        if len(r) == 0:
            continue
        r  = r[-300:][::10]          # very aggressive downsample
        sm = smooth(r).astype(np.float32)
        save_line_plot_pil(
            sm, sname,
            os.path.join(GRAPH_DIR, f"01_stage{i+1}_curve.png"),
            color=colors_rgb[i % len(colors_rgb)]
        )
        del r, sm
        gc.collect()
else:
    print("⚠️  No stage data.")

✅ Saved: ./ppt_graphs\01_stage1_curve.png
✅ Saved: ./ppt_graphs\01_stage2_curve.png
✅ Saved: ./ppt_graphs\01_stage3_curve.png


In [12]:
import gc, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def smooth(y, alpha=0.05):
    if len(y) == 0:
        return np.array([])
    s = [float(y[0])]
    for v in y[1:]:
        s.append(alpha * float(v) + (1 - alpha) * s[-1])
    return np.array(s)

if stage_data:
    os.makedirs(GRAPH_DIR, exist_ok=True)

    for i, (sname, d) in enumerate(stage_data.items()):
        r = np.array(d["rewards"], dtype=np.float32)
        if len(r) == 0:
            print(f"⚠️  No data for {sname}, skipping.")
            continue

        r  = r[-300:][::10]          # keep it small
        sm = smooth(r).astype(np.float32)
        idx = np.arange(len(sm))

        fig, ax = plt.subplots(figsize=(5, 3), dpi=72)
        ax.plot(idx, sm, linewidth=2, color=d["color"], label="Smoothed reward")
        ax.set_title(sname, fontsize=11, fontweight="bold")
        ax.set_xlabel("Episode (downsampled)", fontsize=9)
        ax.set_ylabel("Reward", fontsize=9)
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()

        out = os.path.join(GRAPH_DIR, f"01_stage{i+1}_learning_curve.png")
        fig.savefig(out, dpi=72, bbox_inches="tight")
        plt.close(fig)
        del fig, ax, r, sm, idx
        gc.collect()
        print(f"✅ Saved: {out}")
else:
    print("⚠️  No stage data.")

: 

In [ ]:
# Cell 10 — Graph 01: Per-stage learning curves
# MODIFIED: loads from CSV, last 1000pts [::10], smoothed only, small fig, savefig

def smooth(y, alpha=0.05):
    """Exponential moving average. Returns smoothed array."""
    if len(y) == 0:
        return np.array([])
    smoothed = [float(y[0])]
    for val in y[1:]:
        smoothed.append(alpha * float(val) + (1 - alpha) * smoothed[-1])
    return np.array(smoothed)


if stage_data:
    n_stages = len(stage_data)
    # SMALL figure size to prevent memory crashes
    fig, axes = plt.subplots(1, n_stages, figsize=(4 * n_stages, 3), sharey=False)
    if n_stages == 1:
        axes = [axes]

    for ax, (sname, d) in zip(axes, stage_data.items()):
        # Load from CSV data (already in d["rewards"])
        r = d["rewards"]
        if len(r) == 0:
            ax.set_title(f"{sname}\n(no data)")
            continue

        # Downsample: last 1000 points, then every 10th
        r = r[-1000:][::10]
        sm = smooth(r)
        idx = np.arange(len(sm))

        # Plot ONLY smoothed curve (no raw curve)
        ax.plot(idx, sm, linewidth=2, color=d["color"])
        ax.set_title(sname, fontsize=10)
        ax.set_xlabel("Episode (downsampled)", fontsize=9)
        ax.set_ylabel("Reward (smoothed)", fontsize=9)

    fig.suptitle("Per-Stage Learning Curves", fontsize=12)
    plt.tight_layout()
    # savefig instead of plt.show()
    save_fig(fig, "01_per_stage_learning_curves.png")
else:
    print("⚠️  No stage data — skipping graph 01.")

: 

In [ ]:
# Cell 11 — Graph 02: Combined reward timeline
# MODIFIED: loads from CSV, last 1000pts [::10], smoothed only, small fig, savefig

if stage_data:
    # Small figure
    fig, ax = plt.subplots(figsize=(8, 3))
    offset  = 0

    for sname, d in stage_data.items():
        r = d["rewards"]
        if len(r) == 0:
            continue

        # Downsample: last 1000 points, then every 10th
        r = r[-1000:][::10]
        sm = smooth(r)
        idx = np.arange(len(sm)) + offset

        # Plot ONLY smoothed curve
        ax.plot(idx, sm, color=d["color"], linewidth=2, label=sname)

        # Vertical divider between stages
        if offset > 0:
            ax.axvline(offset, color="#cccccc", linewidth=1, linestyle="--")

        offset += len(sm)

    ax.set_title("Combined Reward Timeline", fontsize=12, fontweight="bold")
    ax.set_xlabel("Episode (downsampled, across stages)", fontsize=9)
    ax.set_ylabel("Reward (smoothed)", fontsize=9)
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    save_fig(fig, "02_combined_reward_timeline.png")
else:
    print("⚠️  No stage data — skipping graph 02.")

In [ ]:
# Cell 12 — Graph 03: Task completion time per stage (Metric 4)
# MODIFIED: loads from CSV, last 1000pts [::10], smoothed only, small fig, savefig

if stage_data:
    n_stages = len(stage_data)
    # Small figure
    fig, axes = plt.subplots(1, n_stages, figsize=(4 * n_stages, 3), sharey=False)
    if n_stages == 1:
        axes = [axes]

    for ax, (sname, d) in zip(axes, stage_data.items()):
        l, c = d["lengths"], d["color"]
        if len(l) == 0:
            ax.set_title(f"{sname}\n(no data)")
            continue

        # Downsample: last 1000 points, then every 10th
        l = l[-1000:][::10]
        sm = smooth(l, alpha=0.05)
        idx = np.arange(len(sm))

        # Plot ONLY smoothed curve
        ax.plot(idx, sm, color=c, linewidth=2)
        ax.set_title(sname, fontsize=10)
        ax.set_xlabel("Episode (downsampled)", fontsize=9)
        ax.set_ylabel("Steps (smoothed)", fontsize=9)

        if len(sm) > 0:
            ax.annotate(
                f"{sm[-1]:.0f} steps",
                xy=(idx[-1], sm[-1]),
                xytext=(-50, 12), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color=c),
                fontsize=8, color=c, fontweight="bold"
            )

    fig.suptitle("Task Completion Time per Stage — Metric 4",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "03_completion_time_per_stage.png")
else:
    print("⚠️  No stage data — skipping graph 03.")

In [ ]:
# Cell 13 — Graph 04: Eval reward & success rate per stage (Metrics 1 & 3)
# MODIFIED: loads from CSV, last 1000pts [::10], smoothed only, small fig, savefig

SUCCESS_THR = {
    "Stage 1 — Reach":      -0.1,
    "Stage 2 — Grasp/Lift": 0.5,
    "Stage 3 — Place":      -50,
}

if stage_data:
    n_stages = len(stage_data)
    # Small figure
    fig, axes = plt.subplots(1, n_stages, figsize=(4 * n_stages, 3))
    if n_stages == 1:
        axes = [axes]

    for ax, (sname, d) in zip(axes, stage_data.items()):
        c        = d["color"]
        eval_log = d.get("eval_log", "")

        if eval_log and os.path.exists(eval_log):
            # Use EvalCallback npz (richer: per-eval snapshot)
            ev      = np.load(eval_log, allow_pickle=True)
            ts      = ev["timesteps"]
            results = ev["results"]          # shape: (n_evals, n_eval_eps)
            mean_r  = results.mean(axis=1)
            thr     = SUCCESS_THR.get(sname, 0)
            sr      = (results > thr).mean(axis=1) * 100

            # Downsample eval data: last 1000 pts [::10]
            ts     = ts[-1000:][::10]
            mean_r = mean_r[-1000:][::10]
            sr     = sr[-1000:][::10]

            # Plot ONLY smoothed
            sm_r  = smooth(mean_r)
            sm_sr = smooth(sr)
            idx   = np.arange(len(sm_r))

            ax2 = ax.twinx()
            ax.plot(idx, sm_r, color=c, linewidth=2, label="Mean reward")
            ax2.plot(idx, sm_sr, color=c, linewidth=1.5, linestyle="--", alpha=0.8,
                     label="Success %")
            ax2.set_ylabel("Success Rate (%)", color=c, fontsize=8)
            ax2.set_ylim(0, 105)

            lines1, labs1 = ax.get_legend_handles_labels()
            lines2, labs2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc="lower right")

        else:
            # Fallback: plot from CSV data (smoothed only)
            r = d["rewards"]
            if len(r) == 0:
                ax.set_title(f"{sname}\n(no data)")
                continue

            # Downsample: last 1000 pts [::10]
            r = r[-1000:][::10]
            thr = SUCCESS_THR.get(sname, 0)

            sm_r = smooth(r)
            idx  = np.arange(len(sm_r))

            # Rolling success rate on downsampled data
            sr = pd.Series((r > thr).astype(float)).rolling(20, min_periods=1).mean().values * 100
            sm_sr = smooth(sr)

            ax2 = ax.twinx()
            # Plot ONLY smoothed
            ax.plot(idx, sm_r, color=c, linewidth=2, label="Reward (smoothed)")
            ax2.plot(idx, sm_sr, color=c, linewidth=1.5, linestyle="--", alpha=0.8,
                     label="Success %")
            ax2.set_ylabel("Success Rate (%)", color=c, fontsize=8)
            ax2.set_ylim(0, 105)

            lines1, labs1 = ax.get_legend_handles_labels()
            lines2, labs2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc="lower right")
            ax.set_xlabel("Episode (downsampled)", fontsize=8)

        ax.set_title(sname, fontsize=10)
        ax.set_ylabel("Eval Reward", color=c, fontsize=8)

    fig.suptitle("Evaluation: Reward & Success Rate per Stage",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    save_fig(fig, "04_eval_reward_success_per_stage.png")
else:
    print("⚠️  No stage data — skipping graph 04.")

In [ ]:
# Cell 14 — Graph 05: Reward distribution boxplot
# MODIFIED: loads from CSV, small fig, savefig

if stage_data:
    # Small figure
    fig, ax = plt.subplots(figsize=(6, 4))
    plot_data, plot_labels, plot_colors = [], [], []

    for sname, d in stage_data.items():
        r = d["rewards"]
        if len(r) > 10:
            # Use last 30% of data (already loaded from CSV — minimal memory)
            tail = r[int(len(r) * 0.7):]
            plot_data.append(tail)
            plot_labels.append(sname.replace(" — ", "\n"))
            plot_colors.append(d["color"])

    if plot_data:
        bp = ax.boxplot(
            plot_data,
            patch_artist  = True,
            notch         = True,
            widths        = 0.4,
            medianprops   = dict(color="white", linewidth=2),
            whiskerprops  = dict(linewidth=1.2),
            capprops      = dict(linewidth=1.2),
            flierprops    = dict(marker="o", markersize=2, alpha=0.3),
        )
        for patch, color in zip(bp["boxes"], plot_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.80)
        for flier, color in zip(bp["fliers"], plot_colors):
            flier.set_markerfacecolor(color)

        ax.set_xticks(range(1, len(plot_labels) + 1))
        ax.set_xticklabels(plot_labels, fontsize=9)
        ax.set_ylabel("Episode Reward (last 30% of training)", fontsize=9)
        ax.set_title("Reward Distribution per Stage — Policy Consistency",
                     fontsize=11, fontweight="bold")

        for i, (data, color) in enumerate(zip(plot_data, plot_colors), 1):
            med = np.median(data)
            ax.text(i, med, f" {med:.2f}", va="center", fontsize=9,
                    color="white", fontweight="bold")
    else:
        ax.text(0.5, 0.5, "Not enough episode data for boxplot",
                ha="center", va="center", transform=ax.transAxes, fontsize=11)

    plt.tight_layout()
    save_fig(fig, "05_reward_distribution_boxplot.png")
else:
    print("⚠️  No stage data — skipping graph 05.")

In [ ]:
# Cell 15 — Final evaluation of the best Stage 3 model

def evaluate_final_model(model_path: str, n_episodes: int = 50):
    env = gym.make(BASE_ENV_ID, render_mode=None)
    env = PlaceRewardWrapper(env)
    mdl = PPO.load(model_path, device=DEVICE)

    successes, distances, steps_list, rewards_list = [], [], [], []

    print(f"⏳ Evaluating {n_episodes} episodes ...")
    for ep in range(n_episodes):
        obs, info = env.reset()
        done, truncated = False, False
        ep_r, ep_s = 0.0, 0

        while not (done or truncated):
            action, _ = mdl.predict(obs, deterministic=True)
            obs, r, done, truncated, info = env.step(action)
            ep_r += r
            ep_s += 1

        successes.append(float(info.get("is_success", 0)))
        rewards_list.append(ep_r)
        steps_list.append(ep_s)

        if "achieved_goal" in obs and "desired_goal" in obs:
            distances.append(float(np.linalg.norm(
                obs["achieved_goal"] - obs["desired_goal"]
            )))

        if (ep + 1) % 10 == 0:
            print(f"   {ep+1}/{n_episodes} ...")

    env.close()
    return {
        "success_rate":  np.mean(successes) * 100,
        "mean_dist":     np.mean(distances) if distances else float("nan"),
        "mean_reward":   np.mean(rewards_list),
        "std_reward":    np.std(rewards_list),
        "mean_steps":    np.mean(steps_list),
        "std_steps":     np.std(steps_list),
        "n":             n_episodes,
    }


final_stage_dir = os.path.join(SAVE_DIR, "stage3")
best_path = os.path.join(final_stage_dir, "best_model.zip")
if not os.path.exists(best_path):
    best_path = os.path.join(SAVE_DIR, CURRICULUM[-1]["save_name"] + ".zip")

if not os.path.exists(best_path):
    print(f"❌ No model found at {best_path}")
    print("   Run training first (Cell 8), or set best_path manually.")
    metrics = None
else:
    print(f"Loading: {best_path}")
    metrics = evaluate_final_model(best_path, n_episodes=50)

    # Save metrics to disk so Cell 16 works after a crash
    metrics_path = os.path.join(LOG_DIR, "final_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"  💾 Metrics saved → {metrics_path}")

    print("\n" + "═" * 52)
    print("  📊  FINAL EVALUATION — Stage 3 Policy")
    print("═" * 52)
    print(f"  Metric 1 — Success Rate    : {metrics['success_rate']:.1f}%")
    print(f"  Metric 2 — Accuracy (dist) : {metrics['mean_dist']:.4f} m")
    print(f"  Metric 3 — Mean Reward     : {metrics['mean_reward']:.2f} ± {metrics['std_reward']:.2f}")
    print(f"  Metric 4 — Steps (avg)     : {metrics['mean_steps']:.1f} ± {metrics['std_steps']:.1f}")
    print("═" * 52)

In [ ]:
# Cell 16 — Graph 06: Final metrics dashboard
# Loads metrics from disk if not in memory (crash-safe)

# Load metrics from disk if needed
if 'metrics' not in dir() or metrics is None:
    metrics_path = os.path.join(LOG_DIR, "final_metrics.json")
    if os.path.exists(metrics_path):
        with open(metrics_path, "r") as f:
            metrics = json.load(f)
        print(f"📂 Loaded metrics from {metrics_path}")
    else:
        print("❌ No metrics found. Run Cell 15 first.")
        metrics = None

if metrics is not None:
    # Small figure
    fig = plt.figure(figsize=(10, 4))
    gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.45)

    metric_cfg = [
        {"label": "Task Success\nRate (%)",   "val": metrics["success_rate"],
         "std": 0,                             "fmt": ".1f",
         "color": STAGE_COLORS["Stage 1 — Reach"]},
        {"label": "Accuracy\n(dist m ↓)",     "val": metrics["mean_dist"] if not np.isnan(metrics["mean_dist"]) else 0,
         "std": 0,                             "fmt": ".4f",
         "color": STAGE_COLORS["Stage 2 — Grasp/Lift"]},
        {"label": "Mean Reward",               "val": abs(metrics["mean_reward"]),
         "std": metrics["std_reward"],         "fmt": ".1f",
         "color": STAGE_COLORS["Stage 3 — Place"]},
        {"label": "Avg Completion\nSteps ↓",  "val": metrics["mean_steps"],
         "std": metrics["std_steps"],          "fmt": ".0f",
         "color": "#7b2d8b"},
    ]

    for i, cfg in enumerate(metric_cfg):
        ax  = fig.add_subplot(gs[0, i])
        ax.bar(
            [0], [cfg["val"]],
            color   = cfg["color"],
            alpha   = 0.85,
            width   = 0.5,
            yerr    = cfg["std"] if cfg["std"] > 0 else None,
            capsize = 8,
            error_kw = dict(elinewidth=2, ecolor="#333")
        )
        label_y = cfg["val"] + (cfg["std"] if cfg["std"] > 0 else 0) + cfg["val"] * 0.04
        ax.text(0, max(label_y, cfg["val"] * 1.05),
                format(cfg["val"], cfg["fmt"]),
                ha="center", va="bottom",
                fontsize=13, fontweight="bold", color=cfg["color"])
        ax.set_title(cfg["label"], fontsize=10, pad=6)
        ax.set_xlim(-0.6, 0.6)
        ax.set_ylim(0, (cfg["val"] + (cfg["std"] if cfg["std"] > 0 else 0)) * 1.35 or 1)
        ax.set_xticks([])
        ax.yaxis.set_visible(False)
        ax.spines["left"].set_visible(False)

    fig.suptitle(
        f"PPO Curriculum — Final Results ({metrics['n']} episodes | Stage 3 Policy)",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    save_fig(fig, "06_final_metrics_dashboard.png")
else:
    print("⚠️  Skipping graph 06 — no metrics available.")

In [ ]:
# Cell 17 — Live GUI Demo (optional — requires a display)
# Skip this cell if running headless / on a server
N_DEMO = 5

if 'best_path' not in dir() or not os.path.exists(best_path):
    print("⚠️  best_path not set. Run Cell 15 first.")
else:
    demo_env   = gym.make(BASE_ENV_ID, render_mode="human")
    demo_env   = PlaceRewardWrapper(demo_env)
    demo_model = PPO.load(best_path, device=DEVICE)

    print(f"🎮 Running {N_DEMO} live episodes — record your screen!")
    print("   Ctrl+C to stop early.\n")

    try:
        for ep in range(N_DEMO):
            obs, info = demo_env.reset()
            done, trunc = False, False
            ep_r, ep_s  = 0.0, 0

            while not (done or trunc):
                action, _ = demo_model.predict(obs, deterministic=True)
                obs, r, done, trunc, info = demo_env.step(action)
                ep_r += r
                ep_s += 1

            ok = info.get("is_success", False)
            print(f"  Ep {ep+1}/{N_DEMO} | reward={ep_r:7.2f} | steps={ep_s} | "
                  f"{'✅ SUCCESS' if ok else '❌ failed'}")

    except KeyboardInterrupt:
        print("⏸  Stopped.")
    finally:
        demo_env.close()
        print("\n✅ Demo closed.")

## 📁 Output Files

| File | What it shows |
|---|---|
| `ppt_graphs/01_per_stage_learning_curves.png` | Each stage converging |
| `ppt_graphs/02_combined_reward_timeline.png` | Full curriculum in one view |
| `ppt_graphs/03_completion_time_per_stage.png` | Episode length trend (Metric 4) |
| `ppt_graphs/04_eval_reward_success_per_stage.png` | Eval snapshots (Metrics 1 & 3) |
| `ppt_graphs/05_reward_distribution_boxplot.png` | Policy consistency |
| `ppt_graphs/06_final_metrics_dashboard.png` | All 4 metrics at a glance |
| `training_log.csv` | Shared log: stage_name, episode_number, reward |
| `logs/training_log_<stage>.csv` | Per-stage detailed log |
| `logs/training_manifest.json` | Stage index for crash-safe reload |